In [ ]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import numpy as np

def single_fighter_scraper(fighter_page_url):
    response = requests.get(str(fighter_page_url))
    html = response.text
    soup = BeautifulSoup(html, "html.parser")
    fighter_soup = soup.find('section', class_ = 'b-statistics__section_details')

    #For record
    heading_soup = soup.find('h2')
    record_unclean = heading_soup.find('span', class_ = 'b-content__title-record').text.strip()
    record = record_unclean[8:]

    #For fight table
    fight_columns = [column.text.strip() for column in fighter_soup.find_all('th')]
    fight_rows = fighter_soup.find_all('tr')

    fight_data = []
    for row in fight_rows:
        cells = row.find_all('td')
        clean_cell = [cell.text.strip() for cell in cells]
        fight_data.append(clean_cell)
    df_fight_table = pd.DataFrame(fight_data, columns = fight_columns)

    #For biostats table
    biostats_table_soup = fighter_soup.find('ul')
    biostats_columns_soup = biostats_table_soup.find_all('i', class_ = 'b-list__box-item-title b-list__box-item-title_type_width')
    biostats_columns = [column.text.strip() for column in biostats_table_soup.find_all('i', class_ = 'b-list__box-item-title b-list__box-item-title_type_width')]

    biostats_data = []
    for i in range(len(biostats_columns)):
        biostats_data.append(biostats_table_soup.find_all('i', class_ = 'b-list__box-item-title b-list__box-item-title_type_width')[i].next_sibling.strip())
    df_biostats_table = pd.DataFrame([biostats_data], columns = biostats_columns)


    #For career statistics table
    career_stat_soup = fighter_soup.find('div', class_="b-list__info-box-left")
    career_stat_columns = [column.text.strip() for column in career_stat_soup.find_all('i')]

    career_stat_data = []
    for i in range(len(career_stat_columns)):
        career_stat_data.append(career_stat_soup.find_all('i')[i].next_sibling.strip())
    df_career_stat = pd.DataFrame([career_stat_data], columns = career_stat_columns)

    return record, df_career_stat, biostats_data, df_fight_table

def single_fight_scraper(fight_url):


    response = requests.get(str(fight_url))
    html = response.text
    soup = BeautifulSoup(html, "html.parser")
    fight_soup = soup.find('div', class_ = 'b-fight-details')


    #For winner/loser/draw info:
    ordered_winner_result = [result.text.strip() for result in fight_soup.find_all('i', class_ = 'b-fight-details__person-status')]
    ordered_fighter_result = [result.text.strip() for result in fight_soup.find_all('a', class_ = 'b-link b-fight-details__person-link')]

    for i in range(2):
        if ordered_winner_result[i] == 'D':
            winner = 'TIE'
            loser = 'TIE'
        elif ordered_winner_result[i] == 'W':
            winner = ordered_fighter_result[i]
        else:
            loser = ordered_fighter_result[i]


    #For fight_details
    fight_details_soup = soup.find('p', class_ = 'b-fight-details__text')
    fight_details_columns = [column.text.strip() for column in fight_details_soup.find_all('i', class_ = 'b-fight-details__label')]

    fight_details_data = []

    fight_details_first_soup = fight_details_soup.find('i', class_ = "b-fight-details__text-item_first")
    fight_details_data.append(fight_details_first_soup.find('i', style="font-style: normal").text.strip())
    labels = fight_details_soup.find_all('i', class_ = 'b-fight-details__label')

    for column in labels:
        fight_details_data.append(column.next_sibling.text.strip())

    fight_details_data.pop(1) #remove blank space
    fight_details_columns.pop(4) #remove referee, since irrelevant
    fight_details_data.pop(4) #removes referee since irrelevant
    df_fight_details = pd.DataFrame([fight_details_data], columns = fight_details_columns)


    #round_by_round table
    rbr_soup = fight_soup.find('tr', class_ = "b-fight-details__table-row") # round by round soup
    rbr_columns = [column.text.strip() for column in rbr_soup] #has '' empty strings
    rbr_columns = [column for column in rbr_columns if column != ''] #removes empty strings

    rbr_fight_data_soup = fight_soup.find_all('section', class_ = "b-fight-details__section js-fight-section")[1]
    rbr_fight_datacell_soup = [column.text.strip() for column in rbr_fight_data_soup.find_all('p', class_ = "b-fight-details__table-text")]

    fighter_one_data = rbr_fight_datacell_soup[::2]
    fighter_two_data = rbr_fight_datacell_soup[1::2]

    df_rbr_total = pd.DataFrame(([fighter_one_data, fighter_two_data]), columns = rbr_columns)

    #aggregate fight table
    #round by round table
    #aggregate significant strikes table
    #round by round sig strikes table

    sections = fight_soup.find_all('section', class_="b-fight-details__section js-fight-section")

    index = [1, 2, 4]
    for i in index:
        section = sections[i]

        rbr_soup = section.find('tr', class_="b-fight-details__table-row")
        rbr_columns = [column.get_text(strip=True) for column in rbr_soup.find_all(['th', 'td'])]

        rbr_data = sections[i].find_all('p', class_="b-fight-details__table-text")
        rbr_data = [column.text.strip() for column in rbr_data]

        fighter_one_data = rbr_data[::2]
        fighter_two_data = rbr_data[1::2]
    
        arr_one = np.array(fighter_one_data)
        arr1 = arr_one.reshape(len(arr_one) // len(rbr_columns), len(rbr_columns))
        df_fighter_one = pd.DataFrame(arr1, columns=rbr_columns)

        arr_two = np.array(fighter_two_data)
        arr2 = arr_two.reshape(len(arr_two) // len(rbr_columns), len(rbr_columns))
        df_fighter_two = pd.DataFrame(arr2, columns=rbr_columns)

        if i == 1:
            df_aggregate_fight_1 = df_fighter_one
            df_aggregate_fight_2 = df_fighter_two
        if i == 2:
            df_rbr_fight_1 = df_fighter_one
            df_rbr_fight_2 = df_fighter_two
        if i == 4:
            df_sig_strikes_rbr_1 = df_fighter_one
            df_sig_strikes_rbr_2 = df_fighter_two


    #Sig strikes total table
    column_soup = soup.find('thead', class_="b-fight-details__table-head")
    data_columns = [column.text.strip() for column in column_soup.find_all('th')]
    
    data_soup = soup.find_all('p', class_ = "b-fight-details__table-text")
    data = [column.text.strip() for column in data_soup]

    df_sig_stikes_total_1 = data[::2]
    df_sig_stikes_total_2 = data[1::2]

    return winner, loser, df_fight_details.head(), df_rbr_total.head(), df_aggregate_fight_1, df_aggregate_fight_2, df_rbr_fight_1, df_rbr_fight_2, df_sig_strikes_rbr_1, df_sig_strikes_rbr_2, df_sig_stikes_total_1, df_sig_stikes_total_2

def fighters_subpage_url_scraper():
    base_fighter_url = 'http://ufcstats.com/statistics/fighters?char=' # + str(letter)
    alphabet = 'abcdefghijklmnopqrstuvwxyz'
    subpage_url_suffice = '&page=all'

    alphabetized_urls = []
    for letter in alphabet:
        alphabetized_urls.append(base_fighter_url + str(letter) + subpage_url_suffice)

    
    return alphabetized_urls

def fighter_url_scraper(subpage_url):

    response = requests.get(subpage_url)
    html = response.text
    soup = BeautifulSoup(html, "html.parser")

    row_soup = soup.find_all('tr', class_ = "b-statistics__table-row")
    fighter_urls = []

    for row in row_soup:
        a_tag = row.find('a')
        if a_tag:
            link = a_tag['href']
            fighter_urls.append(link)

    return fighter_urls

def events_page_scraper():

    url = 'http://ufcstats.com/statistics/events/completed?page=all'
    response = requests.get(url)
    html = response.text
    soup = BeautifulSoup(html, "html.parser")

    soup = soup.find('table', class_ = "b-statistics__table-events")
    row_soup = soup.find_all('a', class_ = 'b-link b-link_style_black')

    event_urls = []
    for row in row_soup:
        link = row.get('href')
        if link:
            event_urls.append(link)
    
    return event_urls

#

In [46]:
subpage_urls_list = fighters_subpage_url_scraper()

all_fighter_urls = []

for i in range(len(subpage_urls_list)):
    all_fighter_urls.append(fighter_url_scraper(subpage_urls_list[i]))
